# Predicting 30-day readmission

This notebook tests whether the supplied admission/discharge variables predict `readmit_30d`. The evaluation is designed around the decision we would make in practice: scoring a **new person** at discharge. Accordingly, all visits from a person stay in the same cross-validation fold.

The notebook compares a prevalence-only baseline with logistic regression and gradient boosting, evaluates discrimination and calibration from out-of-fold predictions, quantifies uncertainty by resampling people, and inspects the most influential logistic-regression features.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

## Load and validate the inputs

`person_key` is an identifier, so it is used only to form patient groups and is never supplied to a model. `length_of_stay_days` and `n_medications` are usable only if prediction happens at discharge; that is the prediction time assumed here.

In [ ]:
DATA_PATH = Path('data.csv')
DICTIONARY_PATH = Path('columns.csv')
TARGET = 'readmit_30d'
GROUP = 'person_key'

data = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)

assert TARGET in data.columns and GROUP in data.columns
assert set(data[TARGET].dropna().unique()) <= {0, 1}
assert not data[TARGET].isna().any(), 'Rows with unknown outcomes need a documented handling rule.'
assert data[GROUP].notna().all(), 'A patient-level split requires person_key on every row.'

display(dictionary)
print(f'{len(data):,} visits from {data[GROUP].nunique():,} people; {data.shape[1] - 2} candidate predictors.')
print(f'Outcome prevalence: {data[TARGET].mean():.1%} ({data[TARGET].sum():,}/{len(data):,}).')
print(f'Duplicate rows: {data.duplicated().sum():,}; missing cells: {data.isna().sum().sum():,}.')

In [ ]:
quality = pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing_n': data.isna().sum(),
    'missing_pct': data.isna().mean().mul(100).round(2),
    'unique_n': data.nunique(dropna=False),
})
display(quality)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
data[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=['#4C78A8', '#E45756'])
axes[0].set(title='Outcome counts', xlabel='Readmitted within 30 days', ylabel='Visits')
data.groupby(GROUP).size().plot.hist(ax=axes[1], bins=range(1, data.groupby(GROUP).size().max() + 2), color='#72B7B2')
axes[1].set(title='Visits per person', xlabel='Number of visits', ylabel='People')
plt.tight_layout()
plt.show()

## Leakage-safe validation and models

Five-fold stratified **grouped** cross-validation gives every row one out-of-fold prediction while keeping each person in exactly one fold. This tests generalization to unseen people. It is not a temporal or external validation.

Postcode is treated as a category even though it is stored as a number. Missing-value handling is inside each model pipeline, so future missingness would not leak information across folds. The same folds are used for all models.

In [ ]:
y = data[TARGET].astype(int)
groups = data[GROUP]
X = data.drop(columns=[TARGET, GROUP])
categorical = ['sex', 'postcode_area']
numeric = [c for c in X.columns if c not in categorical]

linear_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical),
])

dense_preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical),
])

models = {
    'Prevalence baseline': DummyClassifier(strategy='prior'),
    'Logistic regression': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', LogisticRegression(max_iter=2_000)),
    ]),
    'Gradient boosting': Pipeline([
        ('preprocess', dense_preprocessor),
        ('model', HistGradientBoostingClassifier(
            max_iter=250, learning_rate=0.05, l2_regularization=1.0, random_state=RANDOM_STATE
        )),
    ]),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X, y, groups))

# Explicitly verify the central leakage safeguard.
assert all(set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])) for tr, te in folds)

oof = {}
for name, model in models.items():
    oof[name] = cross_val_predict(
        model, X, y, groups=groups, cv=folds, method='predict_proba', n_jobs=-1
    )[:, 1]

results = pd.DataFrame({
    name: {
        'ROC-AUC': roc_auc_score(y, probability),
        'Average precision': average_precision_score(y, probability),
        'Brier score (lower is better)': brier_score_loss(y, probability),
    }
    for name, probability in oof.items()
}).T
display(results.round(3))

## Patient-level uncertainty and diagnostic plots

Confidence intervals below use a patient-level bootstrap: sampled people bring all their visits with them. This preserves within-person dependence. The bootstrap quantifies sampling variation in these out-of-fold predictions; it does not capture dataset shift or optimism from choosing among models on this same validation run.

In [ ]:
def grouped_bootstrap_ci(y_true, probability, group, metric, n_boot=1_000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    unique_groups = np.asarray(pd.unique(group))
    row_indices = {g: np.flatnonzero(np.asarray(group) == g) for g in unique_groups}
    values = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([row_indices[g] for g in sampled])
        yy = np.asarray(y_true)[idx]
        if np.unique(yy).size == 2:
            values.append(metric(yy, np.asarray(probability)[idx]))
    return np.quantile(values, [0.025, 0.975])

ci_rows = []
for name, probability in oof.items():
    for metric_name, metric in [('ROC-AUC', roc_auc_score), ('Average precision', average_precision_score)]:
        estimate = metric(y, probability)
        lo, hi = grouped_bootstrap_ci(y, probability, groups, metric)
        ci_rows.append((name, metric_name, estimate, lo, hi))

uncertainty = pd.DataFrame(ci_rows, columns=['Model', 'Metric', 'Estimate', '2.5%', '97.5%'])
display(uncertainty.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for name, probability in oof.items():
    RocCurveDisplay.from_predictions(y, probability, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y, probability, name=name, ax=axes[1])
    CalibrationDisplay.from_predictions(y, probability, name=name, n_bins=10, strategy='quantile', ax=axes[2])
axes[0].plot([0, 1], [0, 1], '--', color='grey', linewidth=1)
axes[1].axhline(y.mean(), linestyle='--', color='grey', linewidth=1, label='Prevalence')
axes[0].set_title('ROC curves')
axes[1].set_title('Precision–recall curves')
axes[2].set_title('Calibration (10 equal-count bins)')
plt.tight_layout()
plt.show()

## Illustrative operating point

A probability model does not imply a clinical action threshold. To make the trade-off tangible, the next cell shows the logistic model at the threshold whose out-of-fold recall is closest to 70%. This threshold is selected and evaluated on the same data, so the numbers are descriptive—not an unbiased estimate for a deployed policy. A threshold should ultimately be chosen from intervention costs/benefits and then validated on untouched temporal or external data.

In [ ]:
chosen_probability = oof['Logistic regression']
precision, recall, thresholds = precision_recall_curve(y, chosen_probability)
threshold_index = np.argmin(np.abs(recall[:-1] - 0.70))
threshold = thresholds[threshold_index]
prediction = (chosen_probability >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y, prediction).ravel()
operating_point = pd.Series({
    'Threshold': threshold,
    'Recall / sensitivity': tp / (tp + fn),
    'Precision / positive predictive value': tp / (tp + fp),
    'Specificity': tn / (tn + fp),
    'Fraction flagged': prediction.mean(),
    'True positives': tp, 'False positives': fp, 'False negatives': fn, 'True negatives': tn,
})
display(operating_point.to_frame('Value').round(3))

## Interpreting the logistic model

The model is refit on all data only to inspect associations. Numeric coefficients are per one standard deviation; categorical coefficients compare a level with the one-hot parameterization's implicit zero contribution. Coefficients are not causal effects, and their apparent stability is not assessed here.

In [ ]:
final_model = models['Logistic regression'].fit(X, y)
feature_names = final_model.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.Series(
    final_model.named_steps['model'].coef_[0], index=feature_names, name='coefficient'
).sort_values()
most_influential = pd.concat([coefficients.head(8), coefficients.tail(8)]).drop_duplicates()

fig, ax = plt.subplots(figsize=(8, 6))
most_influential.plot.barh(ax=ax, color=np.where(most_influential > 0, '#E45756', '#4C78A8'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Largest fitted logistic coefficients', xlabel='Log-odds coefficient')
plt.tight_layout()
plt.show()
display(coefficients.to_frame().sort_values('coefficient', key=abs, ascending=False).head(15).round(3))

## Conclusion and limitations

The following conclusion is generated from the measured out-of-fold metrics. A useful deployment claim would still require a prespecified use case, comparison with current clinical practice, subgroup/fairness review, temporal or external validation, and prospective monitoring.

In [ ]:
best_name = results['Average precision'].idxmax()
best = results.loc[best_name]
baseline_ap = y.mean()
print(
    f'Answer: yes, the available variables contain predictive signal, but it is modest. '
    f'{best_name} achieved an out-of-fold ROC-AUC of {best["ROC-AUC"]:.3f} and '
    f'average precision of {best["Average precision"]:.3f}, versus an event prevalence '
    f'of {baseline_ap:.3f}. The result supports risk stratification, not reliable individual '
    f'determination or immediate clinical deployment.'
)
print('\nKey limitations:')
print('- No admission/discharge dates are provided, so temporal validation and age-at-visit cannot be constructed.')
print('- Grouped validation measures performance on unseen people from this dataset, not another hospital or time period.')
print('- Model selection, threshold exploration, and evaluation share the same cross-validation predictions.')
print('- Postcode and sex can encode sensitive or structural differences; subgroup performance and intended use need review.')
print('- The label definition and ascertainment window should be audited before operational use.')